# 3. Interaktiv Visualisering (EXTRA)
Plotly-baserade interaktiva plottar för presentationen.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# Läs in det renade datasetet som skapades i steg 1
df = pd.read_csv("../data/cleaned_data.csv")

# Skapa ett deldataset för 2019 respektive 2025
# Vi indexerar på landnamn så att vi enkelt kan slå ihop dem
df_2019 = df[df["Year"] == 2019].set_index("Country name")[["Life evaluation (3-year average)"]]
df_2025 = df[df["Year"] == 2025].set_index("Country name")[["Life evaluation (3-year average)"]]

# Slå ihop 2019 och 2025 på landnamn och beräkna delta (förändring i lyckopoäng)
# Positivt delta = vinnare, negativt delta = förlorare
df_förändring = df_2019.join(df_2025, lsuffix="_2019", rsuffix="_2025")
df_förändring["delta"] = (
    df_förändring["Life evaluation (3-year average)_2025"]
    - df_förändring["Life evaluation (3-year average)_2019"]
)

# Sortera från störst till minst förändring
df_förändring = df_förändring.sort_values("delta", ascending=False)

# Plocka ut topp 10 vinnare och topp 10 förlorare
topp10_vinnare = df_förändring.head(10)
topp10_förlorare = df_förändring.tail(10)

# Bekräfta att datan laddats in korrekt
print(f"Data inläst: {df.shape[0]} rader, {df['Country name'].nunique()} länder")

In [ ]:
# Sätt ihop vinnare och förlorare, sortera så vinnare är överst
df_topp = pd.concat([topp10_vinnare, topp10_förlorare])
df_topp = df_topp.sort_values("delta")

# Gradient: störst absolut delta = mörkast, minst = ljusast
# Stor kontrast mellan ljus och mörk för tydlig gradienteffekt
max_pos = df_topp[df_topp["delta"] > 0]["delta"].max()
max_neg = abs(df_topp[df_topp["delta"] < 0]["delta"].min())

def gradient_färg(delta):
    if delta > 0:
        intensitet = delta / max_pos
        ljus = (180, 255, 180)   # ljusgrön
        mörk = (0, 100, 0)       # mörkgrön
    else:
        intensitet = abs(delta) / max_neg
        ljus = (255, 180, 180)   # ljusröd
        mörk = (140, 0, 0)       # mörkröd
    r = int(mörk[0] + (1 - intensitet) * (ljus[0] - mörk[0]))
    g = int(mörk[1] + (1 - intensitet) * (ljus[1] - mörk[1]))
    b = int(mörk[2] + (1 - intensitet) * (ljus[2] - mörk[2]))
    return f"rgb({r},{g},{b})"

färger = [gradient_färg(d) for d in df_topp["delta"]]

fig = go.Figure(go.Bar(
    x=df_topp["delta"],
    y=df_topp.index,
    orientation="h",
    marker=dict(color=färger),
    text=df_topp["delta"].round(2),
    textposition="outside",
    textfont=dict(color="white")
))

fig.update_layout(
    title=dict(
        text="Topp 10 vinnare & förlorare i lyckopoäng 2019-2025",
        font=dict(color="white", size=16),
        x=0.5
    ),
    xaxis=dict(
        title="Förändring i lyckopoäng (delta)",
        title_font=dict(color="white"),
        tickfont=dict(color="white"),
        gridcolor="#444444",
        zeroline=True,
        zerolinecolor="white",
        zerolinewidth=1
    ),
    yaxis=dict(
        tickfont=dict(color="white"),
        gridcolor="#444444"
    ),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=600
)

fig.show()

In [ ]:
# Utvalda länder för tidslinjen — topp 3 vinnare och botten 3 förlorare
utvalda_länder = ["Viet Nam", "India", "China", "Pakistan", "Lebanon", "Afghanistan"]

# Filtrera datan till enbart dessa länder
df_tidslinje = df[df["Country name"].isin(utvalda_länder)]

# Tydligt skilda nyanser — gröna toner för vinnare, röda/orange toner för förlorare
färg_per_land = {
    "Viet Nam":    "#00e676",
    "India":       "#69f0ae",
    "China":       "#b9f6ca",
    "Pakistan":    "#ff6d00",
    "Lebanon":     "#dd2c00",
    "Afghanistan": "#ff1744",
}

fig = go.Figure()

for land in utvalda_länder:
    df_land = df_tidslinje[df_tidslinje["Country name"] == land].sort_values("Year")
    sista_år = df_land["Year"].max()
    sista_värde = df_land[df_land["Year"] == sista_år]["Life evaluation (3-year average)"].values[0]

    fig.add_trace(go.Scatter(
        x=df_land["Year"],
        y=df_land["Life evaluation (3-year average)"],
        mode="lines+markers",
        name=land,
        showlegend=False,
        line=dict(color=färg_per_land[land], width=2.5),
        marker=dict(size=7)
    ))

    # Lägg till landnamn direkt till höger om sista datapunkten
    fig.add_annotation(
        x=sista_år,
        y=sista_värde,
        text=f"  {land}",
        showarrow=False,
        xanchor="left",
        font=dict(color=färg_per_land[land], size=12)
    )

fig.update_layout(
    title=dict(
        text="Lyckopoängens utveckling 2019–2025 för utvalda länder",
        font=dict(color="white", size=16),
        x=0.5
    ),
    xaxis=dict(
        title="År",
        title_font=dict(color="white"),
        tickfont=dict(color="white"),
        gridcolor="#444444",
        dtick=1,
        range=[2019, 2026]  # extra utrymme för landnamnen
    ),
    yaxis=dict(
        title="Lyckopoäng",
        title_font=dict(color="white"),
        tickfont=dict(color="white"),
        gridcolor="#444444"
    ),
    plot_bgcolor="#2d2d2d",
    paper_bgcolor="#2d2d2d",
    height=500
)

fig.show()

In [ ]:
# Förbered data — delta per land som en platt DataFrame
df_karta = df_förändring[["delta"]].reset_index()
df_karta.columns = ["Country", "delta"]

# Världskarta färgad efter förändring i lyckopoäng
# Grön = förbättring, röd = försämring
fig = px.choropleth(
    df_karta,
    locations="Country",
    locationmode="country names",
    color="delta",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    title="Förändring i lyckopoäng per land 2019-2025",
    labels={"delta": "Förändring (delta)"}
)

fig.update_layout(
    title=dict(font=dict(color="white", size=16), x=0.5),
    paper_bgcolor="#2d2d2d",
    geo=dict(
        bgcolor="#2d2d2d",
        showframe=False,
        showcoastlines=True,
        coastlinecolor="#888888",
        showland=True,
        landcolor="#444444",
        showocean=True,
        oceancolor="#1a1a1a"
    ),
    coloraxis_colorbar=dict(
        title="Delta",
        tickfont=dict(color="white"),
        title_font=dict(color="white")
    ),
    height=550
)

fig.show()